# 1. RAG 평가 개요
- RAG 평가란 RAG 시스템이 주어진 입력에 대해 얼마나 효과적으로 관련 정보를 검색하고, 이를 기반으로 정확하고 유의미한 응답을 생성하는지를 측정하는 과정이다. 
- **평가 요소**
    - **검색 단계 평가**
        - 입력 질문에 대해 검색된 문서나 정보의 관련성과 정확성을 평가.
    - **생성 단계 평가**
        - 검색된 정보를 기반으로 생성된 응답의 품질, 정확성등을 평가.
- **평가 방법**
    - **온/오프라인 평가**
        1. **오프라인 평가**
            - 미리 준비된 데이터셋을 활용하여 RAG 시스템의 성능을 측정한다.
        2. **온라인 평가**
            - 실제 사용자 트래픽과 피드백을 기반으로 시스템의 실시간 성능을 평가한다.
    - **정량적/정성적 평가**
        1. 정량적 평가
            - 자동화된 지표를 사용하여 생성된 텍스트의 품질을 평가한다.
        2. 정성적 평가
            - 전문가나 일반 사용자가 직접 생성된 응답의 품질을 평가하여 주관적인 지표를 평가한다.

# 2. [RAGAS](https://www.ragas.io/)
- RAGAS는 RAG 파이프라인을 **정량적으로 평가하는** 오픈소스 프레임 워크이다. 
- RAGAS 문서: https://docs.ragas.io/en/stable/
## 2.1 설치
- `pip install ragas rapidfuzz`

## 2.2 RAGAS 평가 지표 개요
![ragas_score](figures/ragas_score.png)
- **Generation**
    - llm 모델이 생성한 답변에 대한 평가 지표들.
    - **Faithfulness(신뢰성)**
        -  생성된 답변과 검색된 문서(context)간의 관련성을 평가하는 지표
        -  생성된 답변이 주어진 문맥(context)에 얼마나 충실한지를 평가하는 지표로 할루시네이션에 대한 평가로 볼 수있다.
    - **Answer relevancy(답변 적합성)**
        - 생성된 답변과 사용자의 질문간의 관련성을 평가하는 지표
        - 생성된 답변이 사용자의 질문과 얼마나 관련성이 있는지를 평가하는 지표.
- **Retrieval**
    -  질문에 대해 검색한 문서(context)들에 대한 평가
    -  **Context Precision(문맥 정밀도)**
        -  검색된 문서(context)들 중 질문과 관련 있는 것들이 **얼마나 상위 순위에 위치하는지** 평가하는 지표.
    -  **Context Recall(문맥 재현률)**
        -  검색된 문서(context)가 정답(ground-truth)의 정보를 얼마나 포함하고 있는지 평가하는 지표.
- 이러한 지표들은 RAG 파이프라인의 성능을 다각도로 평가하는 데 활용된다.
![RAGAS_score2](figures/RAGAS_score2.png)

## 2.3 주요 평가지표
### 2.3.1 Generation 평가
- LLM이 생성한 답변에 대한 평가
  
#### 2.3.1.1 Faithfulness (신뢰성)
- 생성된 답변이 얼마나 주어진 검색 문서들(context)를 잘 반영해서 생성되었는지 평가한다. 할루시네이션에 대한 평가라고 할 수 있다. 
- 점수범위: **0 ~ 1** (1에 가까울수록 좋음)
- 답변에 포함된 모든 주장이 context에서 얼마나 추출 가능한지를 확인한다.

##### 2.3.1.1.1평가 방법
1. Answer에서 주장 구문(claim statement)들을 생성(추출)한다. (주장이란, 질문(user input)과 관련된 내용)
    - 예) 
        - **질문**: 한국의 수도는 어디이고 인구는 얼마나 되나요? 
        - **LLM 답변**: 한국의 수도는 서울이고 인구수는 3000만명이다. 
        - **주장(claim)**: 
            1. 한국의 수도는 서울이다.
            2. 인구수는 3000만명이다.
2. 각 주장들을 context로 부터 추론 가능한지 판단한다. 이를 바탕으로 faithfulness 점수를 계산한다.
    - 예)
        - context: 한국은 동아시아에 위치하고 있는 나라다. 한국의 수도는 서울이다. .... 한국의 인구는 5000만명이고 서울에 1000만이 살고 있다.
        - 위 context에서 추론 가능한 주장: 
            - 한국의 수도는 서울이다. -> context에서 추론가능한 주장.
            - 한국의 인구는 3000만명이다. -> context에서 추론 불가능한 주장.
3. **Faithfulness score** 를 계산한다. 총 주장 수 중에서 context로 부터 추론가능한 주장의 개수.    
    - 예)
        - Faithfulness Score = $\cfrac{1}{2} = 0.5$ (두 개의 주장 중 한 개의 주장만 context에서 유추할 수있다.)
    - LLM 답변에서 주장을 추출 하는 것과 각 주장이 context에서 추론 가능한 지를 판단하는 것은 LLM 을 활용한다.
- 공식
    $$
    \text{Faithfulness Score}\;=\;\cfrac{\text{주어진\;context\;에서\;추론할\;수\;있는\;주장의\;개수}}{\text{총\;주장\;개수}}
    $$

### 2.3.2 Answer relevancy (답변 적합성)
- 생성된 답변이 질문(user input)에 얼마나 잘 부합하는 지를 평가한다.
- 점수 범위: -1~1 (1에 가까울수록 좋음)
- LLM이 생성한 답변을 기반으로 질문들을 생성한다. 이렇게 생성한 질문들과 실제 질문(user input) 간의 유사도를 측정한다.

#### 2.3.2.1 평가방법
1. LLM이 생성한 답변을 기반으로 질문들을 생성한다.
    - 예) 
        - **LLM** 답변: 한국의 수도는 서울이고 인구수는 3000만명이다. 
        - **생성된 질문**: 
            1. 한국의 수도는 어디이고 인구는 얼마나 되나요?
            2. 한국의 수도는 어디인가요?
            3. 한국의 인구는 얼마나 되나요?
2. 실제 질문과 생성한 질문간의 코사인 유사도를 측정한다. 그 평균이 최종 점수가 된다.
    - 예)
        - **실제 질문**: 한국의 수도는 어디이고 인구는 얼마나 되나요?
        - **생성된 질문**: 
            1. 한국의 수도는 어디이고 인구는 얼마나 되나요?
            2. 한국의 수도는 어디인가요?
            3. 한국의 인구는 얼마나 되나요?
- 공식
  $$
    \cfrac{1}{N} \sum_{i=1}^{N} \text{cosine\_similarity}(q_{\text{user}_{_i}}, q_{\text{generated}})
  $$

## 2.4 Retrieval 평가
Vector store에서 검색한 context에 대한 평가

### 2.4.1 Context Precision
- 검색된 문서(context)들 중 질문과 관련 있는 것들이 얼마나 **상위 순위**에 있는 지 평가.
- 점수 범위: 0~1 (1에 가까울수록 좋음)


#### 2.4.1.1평가방법

- 공식
$$
 \text{Context\;Precision@K} = \frac{\sum_{k=1}^{K} \left( \text{Precision@k} \times v_k \right)}{\ 상위\;K개\;결과에서의\;관련\;항목\;수}
$$
$$
 \text{Precision@k} = \frac{\text{True\;positive@k}}{(\text{True\;positive@k} + \text{False\;positive@k})} \\
$$
- $\text{Precision@k}$: 개별 문서에 대한 Precision
- K: context 의 개수(chuck 수)
- $v_k$: 관련성 여부로 0 또는 1. (0: 관련 없음, 1: 관련 있음)

#### 2.4.1.2 예시
- 질문과 context 관련성의 예
    - 질문: 한국의 수도는 어디이고 인구는 얼마나 되나요?
    - **높은 정밀도 context들**: 질문과 직접적인 관련이 있는 문서들
        - 한국의 수도는 서울이고 인구는 5000만명 입니다. 
        - 한국의 수도는 서울입니다.
        - 한국은 동아시아에 위치해 있는 국가로 수도는 서울입니다.
        - 한국의 인구는 5000만명 입니다.
    - **낮은 정밀도 context**: 한국과 관련있어 검색될 수 있지만 질문과 직접적 관련이 없다. 
        - 한국은 동아시아에 위치한 국가입니다.
        - 한국의 K-pop은 전 세계적으로 유명합니다.
        - 비빔밥, 불고기는 한국의 대표적인 음식입니다.
    - **높은 정밀도의 context이 상위 순위에 위치했으면 높은 점수를 받는다.**

- 점수 계산 예:
    - **상위 5개의 검색 결과 중 1번째, 3번째, 4번째 문서가 관련이 있다고 가정하자.**
    - **Precision@K 계산**
        ```bash
            Precision@1 = 1/1 = 1.0    # True positive@1/(True positive@1 + False positive@1).  1/1(1번 문서 계산 시에는 1개 문서만 있으므로 분모가 1이 된다.)
            Precision@2 = 1/2 = 0.5
            Precision@3 = 2/3 ≈ 0.67    
            Precision@4 = 3/4 = 0.75
            Precision@5 = 3/5 = 0.6
        ```
    - **vk의 값**
        - 1번째: $v_1 = 1$ - 관련있음
        - 2번째: $v_2 = 0$ - 관련없음
        - 3번째: $v_3 = 1$ - 관련있음
        - 4번째: $v_4 = 1$ - 관련있음
        - 5번째: $v_5 = 0$ - 관련없음

    - **Context Precision@5**
        $$
        \text{Context\;Precision@5} = \frac{(1.0 \times 1) + (0.5 \times 0) + (0.67 \times 1) + (0.75 \times 1) + (0.6 \times 0)}{3} = \frac{1.0 + 0 + 0.67 + 0.75 + 0}{3} ≈ 0.807
        $$

### 2.4.2 Context Recall (문맥 재현률)
- 검색된 문서(context)가 얼마나 정답(ground-truth)의 정보를 포함있는 지 평가하는 지표
- 점수 범위: 0~1 (1에 가까울수록 좋음)
- **정답(ground truth)의 각 주장(claim)이 검색된 context와 얼마나 일치**하는지 계산함.

#### 2.4.2.1 평가방법
1. 정답에서 주장(claim)들을 생성(추출)한다.
    - 예) 
        - **정답**: 한국의 수도는 서울이고 인구수는 5000만명이다. 
        - **주장(claim)**: 
            1. 한국의 수도는 서울이다.
            2. 인구수는 5000만명이다.
2. 각 주장(claim)의 정보를 검색된 contexts에서 찾을 수 있는지 판별한다. 이를 바탕으로 context recall 점수를 계산한다.
    - 예)
        - context: 한국은 동아시아에 위치하고 있는 나라다. 한국의 수도는 서울이다.
        - 위 context에서 추론 가능한 주장: 
            - 한국의 수도는 서울이다. -> context에서 찾을 수 있다.
            - 한국의 인구는 5000만명이다. -> context에서 찾을 수 없다.
3. **Context Recall Score** 를 계산한다. 총 주장 수 중에서 context로 부터 찾을 수 있는 주장의 개수.
    - 예)
        - Context Recall Score = $\cfrac{1}{2} = 0.5$ (두 개의 주장 중 한 개의 주장만 context에서 찾을 수 있다.)

- 공식
    $$
    \text{Context Recall Score}\;=\;\cfrac{\text{GT의\;주장\;중\;주어진\;context\;에서\;찾을\;수\;있는\;주장의\;개수}}{\text{GT의\;총\;주장\;개수}}
    $$ 

# 3. RAGAS 평가 실습

In [19]:
# !uv pip install ragas rapidfuzz
# 설치 후 커널 재시작

In [20]:
# Neo4j를 로컬에서 띄우는 예시 (law_pdfs_to_neo4j_pipeline.ipynb 에서 이미 적재된 DB에 연결만 하면 되므로,
# Neo4j 컨테이너가 이미 떠 있다면 이 셀은 실행할 필요 없습니다)
# docker run -p 7474:7474 -p 7687:7687 -v neo4j_data:/data -e NEO4J_AUTH=neo4j/your_password neo4j:latest


## 3.1 Neo4j 연결 및 Retriever 생성
앞서 `law_pdfs_to_neo4j_pipeline.ipynb`로 적재해둔 Neo4j DB에 연결만 하고, text-to-Cypher + 벡터 폴백 방식의 retriever를 만듭니다.

In [21]:
##################################################
# Neo4j 연결 및 retriever 생성
##################################################
import os
import re
from openai import OpenAI
from neo4j import GraphDatabase
from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_core.runnables import RunnableLambda

load_dotenv()

NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")
NEO4J_DATABASE = os.getenv("NEO4J_DATABASE", "lawdb")  # law_pdfs_to_neo4j_pipeline.ipynb 와 동일 기본값

CHAT_MODEL = "gpt-5.4-mini"
EMBEDDING_MODEL = "text-embedding-3-large"  # 적재 시 사용한 임베딩 모델과 반드시 동일해야 함

openai_client = OpenAI()
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))

# 연결 및 적재 상태 확인
with driver.session(database=NEO4J_DATABASE) as session:
    article_count = session.run("MATCH (a:ARTICLE) RETURN count(a) AS c").single()["c"]
    print(f"현재 Neo4j에 적재된 ARTICLE 노드: {article_count}개")
    if article_count == 0:
        print("⚠️ 적재된 데이터가 없습니다. law_pdfs_to_neo4j_pipeline.ipynb를 먼저 실행해주세요.")

FORBIDDEN_KEYWORDS = {"CREATE", "DELETE", "SET", "MERGE", "REMOVE", "DROP", "DETACH"}

# [수정] 관계명 REFERENCE -> REFERENCES
# (law_pdfs_to_neo4j_pipeline.ipynb의 load_references()가 실제로 만드는 관계는
#  MERGE (src)-[:REFERENCES]->(tgt) 이므로 스키마 설명도 이와 일치시켜야 합니다.
#  불일치 시 LLM이 REFERENCE로 Cypher를 짜면 에러 없이 조용히 빈 결과만 나옵니다.)
SCHEMA_DESCRIPTION = """
그래프 DB 스키마:
1. 노드 라벨:
  - LAW (법률/법령 메타데이터)
    * 속성: id (string, 법령명), name (string), law_type (string), promulgation_date (string), effective_date (string)
  - ARTICLE (법령 조문)
    * 속성: id (string, 형식 "법령명::제N조"), name (string, 조문제목), description (string, 본문),
             original_id (string, 예: "제8조"), original_id_normalized (string, 예: "8", "5-2"), law_id (string)

2. 관계(Relationships):
  - (:LAW)-[:CONTAINS]->(:ARTICLE) : 법률이 특정 조문을 포함함.
  - (:ARTICLE)-[:REFERENCES]->(:ARTICLE) : 하나의 조문이 다른 조문을 참조/인용함.

벡터 인덱스: 'article_vector_index' (ARTICLE 노드의 embedding 속성 기준, 의미 검색용, dim=3072, cosine)
"""


class Neo4jRetriever:
    """사용자 질문을 받아 Neo4j(text-to-Cypher + 벡터 폴백)에서 참조 조문을 가져오는 retriever"""

    def __init__(self, client: OpenAI, driver, database: str):
        self.client = client
        self.driver = driver
        self.database = database

    def _is_safe_cypher(self, query: str) -> bool:
        upper = query.upper()
        return not any(re.search(rf"\b{kw}\b", upper) for kw in FORBIDDEN_KEYWORDS)

    def _generate_cypher(self, user_query: str) -> str:
        system_prompt = f"""
당신은 Neo4j Cypher 쿼리를 작성하는 전문가입니다.
아래 그래프 스키마만을 근거로, 사용자 질문에 답하기 위한 최적의 Cypher 쿼리를 작성하세요.

{SCHEMA_DESCRIPTION}

쿼리 작성 가이드라인:
1. 오직 읽기(MATCH) 쿼리만 작성하세요. (CUD 명령어 절대 금지)
2. [특정 조문 지정 질문]: 법령명/조번호가 명시된 경우, 해당 ARTICLE과 REFERENCES로 연결된 조문까지 함께 가져오세요.
3. [주제/의미 검색 질문]: 아래 벡터 검색 형태를 사용하세요:
   CALL db.index.vector.queryNodes('article_vector_index', $top_k, $query_embedding)
   YIELD node, score
   RETURN node.id AS id, node.name AS name, node.description AS description, score
   ORDER BY score DESC
4. RETURN문에는 반드시 'id', 'name', 'description', 'score' 별칭을 사용하세요.
5. 오직 Cypher 쿼리 텍스트만 출력하세요. 마크다운 코드블록이나 설명은 절대 포함하지 마세요.
"""
        response = self.client.chat.completions.create(
            model=CHAT_MODEL,
            temperature=0,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_query},
            ],
        )
        cypher = response.choices[0].message.content.strip()
        cypher = re.sub(r'^```(cypher)?\s*|\s*```$', '', cypher, flags=re.MULTILINE).strip()
        return cypher

    def _vector_fallback(self, user_query: str, top_k: int) -> list[dict]:
        embedding = self.client.embeddings.create(
            model=EMBEDDING_MODEL, input=[user_query]
        ).data[0].embedding

        cypher = """
        CALL db.index.vector.queryNodes('article_vector_index', $top_k, $query_embedding)
        YIELD node, score
        RETURN node.id AS id, node.name AS name, node.description AS description, score
        ORDER BY score DESC
        """
        with self.driver.session(database=self.database) as session:
            results = session.run(cypher, top_k=top_k, query_embedding=embedding)
            return [dict(r) for r in results]

    def retrieve(self, user_query: str, top_k: int = 5) -> list[dict]:
        cypher = self._generate_cypher(user_query)

        if not self._is_safe_cypher(cypher):
            return self._vector_fallback(user_query, top_k)

        params = {"top_k": top_k}
        if "query_embedding" in cypher:
            params["query_embedding"] = self.client.embeddings.create(
                model=EMBEDDING_MODEL, input=[user_query]
            ).data[0].embedding

        try:
            with self.driver.session(database=self.database) as session:
                results = session.run(cypher, **params)
                rows = [dict(r) for r in results]
        except Exception:
            return self._vector_fallback(user_query, top_k)

        if not rows:
            return self._vector_fallback(user_query, top_k)

        return rows


neo4j_retriever = Neo4jRetriever(client=openai_client, driver=driver, database=NEO4J_DATABASE)


def _rows_to_documents(rows: list[dict]) -> list[Document]:
    """Neo4jRetriever가 반환하는 list[dict]를 체인이 기대하는 list[Document]로 변환"""
    return [
        Document(
            page_content=row.get("description", ""),
            metadata={"id": row.get("id"), "name": row.get("name"), "score": row.get("score")},
        )
        for row in rows
    ]


# LCEL 체인에서 retriever 자리를 그대로 대체할 수 있도록 Runnable로 감쌈
retriever = RunnableLambda(lambda query: _rows_to_documents(neo4j_retriever.retrieve(query, top_k=5)))


현재 Neo4j에 적재된 ARTICLE 노드: 1697개


In [22]:
################################################################################
# 평가할 RAG Chain
################################################################################

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document
from langchain_openai import ChatOpenAI

prompt_txt = """<instruction>
당신은 정보제공을 목적으로하는 유능한 AI Assistant 입니다.
주어진 context의 내용을 기반으로 질문에 답변을 합니다.
Context에 질문에 대한 명확한 정보가 있는 경우 그것을 바탕으로 답변을 합니다.
Context에 질문에 대한 명확한 정보가 없는 경우 "정보가 부족해 답을 할 수없습니다." 라고 답합니다.
절대 추측이나 일반 상식을 바탕으로 답을 하거나 Context 없는 내용을 만들어서 답변해서는 안됩니다.
</instruction>
<context>
{context}
</context>
<question>
{query}
</question>
"""
prompt = ChatPromptTemplate.from_template(template=prompt_txt)

model = ChatOpenAI(model="gpt-5.4-mini")
parser = StrOutputParser()


def format_docs_for_prompt(documents: list[Document]) -> str:
    """LLM 프롬프트({context})용: 조문 id/제목을 헤더로 표기해서 하나의 문자열로 합침"""
    parts = []
    for doc in documents:
        meta = doc.metadata
        header = f"[{meta.get('id', '')} {meta.get('name', '')}]"
        parts.append(f"{header}\n{doc.page_content}")
    return "\n\n---\n\n".join(parts)


def format_docs_for_ragas(documents: list[Document]) -> list[str]:
    """RAGAS 평가용: 검색된 문서 각각의 내용을 list[str]로 추출"""
    return [doc.page_content for doc in documents]


def build_prompt_input(state: dict) -> dict:
    """검색 결과({'documents':..., 'query':...})를 prompt 입력({'context':str, 'query':str})으로 변환"""
    return {
        "context": format_docs_for_prompt(state["documents"]),
        "query": state["query"],
    }


# 1단계: 질문으로 검색 → 원본 Document 리스트 + 질문 보존 (retriever는 여기서 1회만 호출됨)
retrieve_step = {
    "documents": retriever,
    "query": RunnablePassthrough(),
}

# RAG 체인 -> RAG 평가 데이터셋을 만드는 RAG Chain -> 최종응답: LLM 응답(str), 검색한 문서들(list[str]), 질문(str)
chain = (
    RunnablePassthrough()  # dict | dict 가 파이썬 dict 병합 연산으로 취급되는 것을 막기 위해 필요
    | retrieve_step
    | {
        "response": RunnableLambda(build_prompt_input) | prompt | model | parser,
        "retrieved_context": RunnableLambda(lambda state: format_docs_for_ragas(state["documents"])),
        "query": RunnableLambda(lambda state: state["query"]),
    }
)


In [23]:
res = chain.invoke("군인연금법 제8조는 무엇인가요?")


In [24]:
res["response"]

'군인연금법 제8조는 **“급여 사유의 확인 및 급여의 결정”**에 관한 조항입니다.  \n구체적으로는,\n\n- **제7조에 따른 급여**는\n- **그 급여를 받을 권리를 가진 사람의 청구에 따라**\n- **국방부장관이 결정하여 지급한다**고 규정하고 있습니다.'

In [25]:
res["retrieved_context"]

['제7조에 따른 급여는 그 급여를 받을 권리를 가진 사람의 청구에 따라 국방부\n장관이 결정하여 지급한다.\n\n법제처                                                            4                                                       국가법령정보센터\n군인연금법',
 '이 법에 따른 급여의 종류는 다음 각 호와 같다.\n1. 퇴직급여\n가. 퇴역연금\n나. 퇴역연금일시금\n다. 퇴역연금공제일시금\n라. 퇴직일시금\n2. 퇴직유족급여\n가. 퇴역유족연금\n나. 퇴역유족연금부가금\n다. 퇴역유족연금특별부가금\n라. 퇴역유족연금일시금\n마. 퇴직유족일시금\n3. 퇴직수당']

# 4. RAGAS 를 이용해 평가를 위한 합성 데이터 셋 만들기

- 평가 데이터셋 구성
  - `user_input`: 사용자 질문
  - `retrieved_contexts`: Vectorstore에서 검색한 context
  - `response`: LLM의 응답
  - `reference`: 정답

## 4.1 TestsetGenerator
- **문서(retrieved_contexts)를 기준**으로 **질문**, **정답** 을 생성한다.
- 평가할 LLM으로 생성된 질문을 넣어 답변을 추출하여 데이터셋을 구성한다.


> **주의**
> - TestsetGenerator import 시 `No Module named langchain_community.chat_models.vertexai` Error 발생 
> - RAGAS와 langchain-community의 버전 호환성 문제 때문에 발생한다.
> - 해결
>   1. langchain_google_vertexai 설치
>       - `!uv pip install langchain_google_vertexai`
>   2. `.venv\Lib\site-packages\langchain_community\chat_models` 디렉토리 아래 `vertexai.py` 파일을 만들고 아래 코드를 > 넣는다.
>    ```python
>       try:
>          from langchain_google_vertexai import ChatVertexAI
>       except ImportError:
>           class ChatVertexAI:
>               def __init__(self, *args, **kwargs):
>                   raise ImportError(
>                       "ChatVertexAI requires langchain-google-vertexai. "
>                       "Install with: pip install langchain-google-vertexai"
>                   )
>    ```

In [26]:
# !uv pip install langchain_google_vertexai

In [27]:
# 주피터노트북 환경에서 비동기적 처리 위해 
# script(.py) 로 작성할 경우는 필요 없다.

import nest_asyncio
nest_asyncio.apply()

In [28]:
from ragas.testset import TestsetGenerator

In [29]:
#
# testset -> Context들(문서들) - [질문 - 정답답변 + (Retriever가 찾은 문서 + LLM 응답: chain에서 생성)]
# 1. Context(문서들)을 추출 - Neo4j에서 ARTICLE 노드를 랜덤 샘플링 -> TestsetGenerator -> 질문과 정답 답변 생성

with driver.session(database=NEO4J_DATABASE) as session:
    total_docs = session.run("MATCH (a:ARTICLE) RETURN count(a) AS c").single()["c"]

    # rand() 정렬로 랜덤 k(5)개 샘플링 (APOC 없이도 동작)
    results = session.run(
        """
        MATCH (a:ARTICLE)
        RETURN a.id AS id, a.name AS name, a.description AS description
        ORDER BY rand()
        LIMIT $k
        """,
        k=10,
    )
    sample_docs = [dict(r) for r in results]

# description(조문 본문)만 추출해서 list[str]
docs = [d["description"] for d in sample_docs]


In [30]:
total_docs

1697

In [31]:
sample_docs

[{'id': '병역법::제29조',
  'name': '사회복무요원의 소집 등',
  'description': '① 지방병무청장은 사회복무요원 소집 순서가 결정된 사람을 대상으로 복무기관을 정\n하여 사회복무요원을 소집한다. 다만, 제26조제4항에 따라 선발한 사회복무요원은 복무기관과 복무분야를 정하여\n따로 소집할 수 있다. <개정 2013. 6. 4.>\n② 병무청장은 사회복무요원 소집이 연기된 사람으로서 그 사유가 없어진 사람 등 대통령령으로 정하는 사람에 대\n하여는 제1항에도 불구하고 지방병무청장으로 하여금 따로 사회복무요원 소집을 하게 할 수 있다.<개정 2013. 6.\n4.>\n③ 제1항과 제2항에 따라 사회복무요원으로 소집된 사람에 대하여는 제55조에 따른 군사교육소집을 하며, 그 군사\n교육소집 기간은 복무기간에 산입한다.<개정 2013. 6. 4., 2016. 5. 29.>\n④ 지방병무청장은 복무기관의 장이 사회복무요원의 임무 부여에 활용할 수 있도록 사회복무요원이 다음 각 호의\n범죄로 인하여 형의 선고를 받은 경우 관련된 정보를 그 사회복무요원이 복무할 복무기관의 장에게 제공할 수 있으\n며, 정보 제공 절차 및 제공되는 정보의 범위는 병무청장이 정한다.<신설 2020. 12. 22., 2024. 1. 9.>\n1. 「아동ㆍ청소년의 성보호에 관한 법률」 제2조제2호, 제3호 및 제3호의2에 따른 범죄\n2. 「개인정보 보호법」 제70조부터 제73조까지에 따른 범죄\n3. 「특정강력범죄의 처벌에 관한 특례법」 제2조의 특정강력범죄\n4. 「형법」 제257조, 제258조, 제258조의2, 제259조부터 제265조까지에 따른 상해와 폭행의 죄, 제283조부터 제\n286조까지에 따른 협박의 죄\n5. 「마약류 관리에 관한 법률」 제58조, 제58조의2, 제59조부터 제64조까지, 제65조의2에 따른 범죄\n[전문개정 2009. 6. 9.]\n[제목개정 2013. 6. 4., 2020. 12. 22.]'},
 {'id': '병역법 시행령:

In [32]:
docs

['① 지방병무청장은 사회복무요원 소집 순서가 결정된 사람을 대상으로 복무기관을 정\n하여 사회복무요원을 소집한다. 다만, 제26조제4항에 따라 선발한 사회복무요원은 복무기관과 복무분야를 정하여\n따로 소집할 수 있다. <개정 2013. 6. 4.>\n② 병무청장은 사회복무요원 소집이 연기된 사람으로서 그 사유가 없어진 사람 등 대통령령으로 정하는 사람에 대\n하여는 제1항에도 불구하고 지방병무청장으로 하여금 따로 사회복무요원 소집을 하게 할 수 있다.<개정 2013. 6.\n4.>\n③ 제1항과 제2항에 따라 사회복무요원으로 소집된 사람에 대하여는 제55조에 따른 군사교육소집을 하며, 그 군사\n교육소집 기간은 복무기간에 산입한다.<개정 2013. 6. 4., 2016. 5. 29.>\n④ 지방병무청장은 복무기관의 장이 사회복무요원의 임무 부여에 활용할 수 있도록 사회복무요원이 다음 각 호의\n범죄로 인하여 형의 선고를 받은 경우 관련된 정보를 그 사회복무요원이 복무할 복무기관의 장에게 제공할 수 있으\n며, 정보 제공 절차 및 제공되는 정보의 범위는 병무청장이 정한다.<신설 2020. 12. 22., 2024. 1. 9.>\n1. 「아동ㆍ청소년의 성보호에 관한 법률」 제2조제2호, 제3호 및 제3호의2에 따른 범죄\n2. 「개인정보 보호법」 제70조부터 제73조까지에 따른 범죄\n3. 「특정강력범죄의 처벌에 관한 특례법」 제2조의 특정강력범죄\n4. 「형법」 제257조, 제258조, 제258조의2, 제259조부터 제265조까지에 따른 상해와 폭행의 죄, 제283조부터 제\n286조까지에 따른 협박의 죄\n5. 「마약류 관리에 관한 법률」 제58조, 제58조의2, 제59조부터 제64조까지, 제65조의2에 따른 범죄\n[전문개정 2009. 6. 9.]\n[제목개정 2013. 6. 4., 2020. 12. 22.]',
 '① 국방부장관은 법 제58조 및 제59조에 따라 의무ㆍ법무ㆍ군종ㆍ수\n의 분야의 현역장교(의무ㆍ법무ㆍ군종ㆍ수의사관후보생을 포함하며, 이하 “의

In [33]:
# testset 생성
from ragas.testset import TestsetGenerator
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
# testsetgenerator 는 gpt-5 이후 버전은 사용할 수 없다.
## Langchain의 모델과 Embedding 모델 -> ragas에서 사용할 수 있도록 변환(wrapping)
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-large"))
generator = TestsetGenerator(
    llm=generator_llm,
    embedding_model=generator_embeddings,
    llm_context="""
- 군 관련 법령(공무원보수규정, 군인사법, 병역법, 군인연금법 등)에 대해 사람들이 궁금해 할 만한 질문들을 생성한다.
- 특정 조문 번호나 법령명을 명시한 질문과, 주제/상황 기반 질문을 골고루 섞는다.
- 데이터셋은 반드시 한국어로 작성한다.
- 데이터셋은 JSON 문법을 지켜서 작성한다. 특히 구두점은 꼭 지켜야 한다.
- 생성된 내용이나 Document에 JSON문법에 맞지 않는 표현이 있으면 반드시 수정해서 처리한다.
"""
    # 질문/답변을 생성할 때 LLM에게 전달할 system Prompt를 설정
)


C:\Users\Playdata\AppData\Local\Temp\ipykernel_30664\1892202356.py:8: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1"))
C:\Users\Playdata\AppData\Local\Temp\ipykernel_30664\1892202356.py:9: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-large"))


In [34]:
testset = generator.generate_with_chunks(
    docs, testset_size=10, # Context 내용 테스트셋 개수(질문 답변 개수)
)

Applying CustomNodeFilter:   0%|          | 0/10 [00:00<?, ?it/s]Node ecee4a6d-8822-4178-8595-aa7d908c48da does not have a summary. Skipping filtering.
Node 3115e51f-b2a1-4e8e-bc86-9dc0eee29d75 does not have a summary. Skipping filtering.
Node dbd4c96a-06cf-41a3-82da-10868e9ff4a2 does not have a summary. Skipping filtering.
Node 7104e305-0022-4907-bc6b-75f231713c7a does not have a summary. Skipping filtering.
Node 6955aba5-b804-4196-983c-c8ebbb57c6af does not have a summary. Skipping filtering.
Node ae63e9cd-b0ea-4417-9b12-91a17f0fb63f does not have a summary. Skipping filtering.
Node 843d63c4-e67f-431f-aa90-6baafacab13f does not have a summary. Skipping filtering.
Node d282f520-48ff-4e7c-a159-fde1fac2c4de does not have a summary. Skipping filtering.
Node 530b6432-4ae3-4aaa-b113-13eaeefaba9f does not have a summary. Skipping filtering.
Node a4485e34-14f3-437a-ae7b-a068bdafea49 does not have a summary. Skipping filtering.
Applying OverlapScoreBuilder: 100%|██████████| 1/1 [00:00<00:00, 

In [35]:
testset

Testset(samples=[TestsetSample(eval_sample=SingleTurnSample(user_input='대통령령에 따라 사회복무요원 소집이 연기된 사람이 다시 소집될 수 있는 경우는 무엇인가요?', retrieved_contexts=None, reference_contexts=['① 지방병무청장은 사회복무요원 소집 순서가 결정된 사람을 대상으로 복무기관을 정\n하여 사회복무요원을 소집한다. 다만, 제26조제4항에 따라 선발한 사회복무요원은 복무기관과 복무분야를 정하여\n따로 소집할 수 있다. <개정 2013. 6. 4.>\n② 병무청장은 사회복무요원 소집이 연기된 사람으로서 그 사유가 없어진 사람 등 대통령령으로 정하는 사람에 대\n하여는 제1항에도 불구하고 지방병무청장으로 하여금 따로 사회복무요원 소집을 하게 할 수 있다.<개정 2013. 6.\n4.>\n③ 제1항과 제2항에 따라 사회복무요원으로 소집된 사람에 대하여는 제55조에 따른 군사교육소집을 하며, 그 군사\n교육소집 기간은 복무기간에 산입한다.<개정 2013. 6. 4., 2016. 5. 29.>\n④ 지방병무청장은 복무기관의 장이 사회복무요원의 임무 부여에 활용할 수 있도록 사회복무요원이 다음 각 호의\n범죄로 인하여 형의 선고를 받은 경우 관련된 정보를 그 사회복무요원이 복무할 복무기관의 장에게 제공할 수 있으\n며, 정보 제공 절차 및 제공되는 정보의 범위는 병무청장이 정한다.<신설 2020. 12. 22., 2024. 1. 9.>\n1. 「아동ㆍ청소년의 성보호에 관한 법률」 제2조제2호, 제3호 및 제3호의2에 따른 범죄\n2. 「개인정보 보호법」 제70조부터 제73조까지에 따른 범죄\n3. 「특정강력범죄의 처벌에 관한 특례법」 제2조의 특정강력범죄\n4. 「형법」 제257조, 제258조, 제258조의2, 제259조부터 제265조까지에 따른 상해와 폭행의 죄, 제283조부터 제\n286조까지에 따른 협박의 죄\n5. 「마약류 관리에 관한 법률」 제58조, 제58조의2

In [36]:
sample1 = testset.samples[0].eval_sample
print("사용자질문:", sample1.user_input)
print("Context:", sample1.reference_contexts)
print("생성된 답변(정답):", sample1.reference)
print("평가대상 RAG의 답변:", sample1.response)
print("평가대상 RAG가 검색한 Context:", sample1.retrieved_contexts)


사용자질문: 대통령령에 따라 사회복무요원 소집이 연기된 사람이 다시 소집될 수 있는 경우는 무엇인가요?
Context: ['① 지방병무청장은 사회복무요원 소집 순서가 결정된 사람을 대상으로 복무기관을 정\n하여 사회복무요원을 소집한다. 다만, 제26조제4항에 따라 선발한 사회복무요원은 복무기관과 복무분야를 정하여\n따로 소집할 수 있다. <개정 2013. 6. 4.>\n② 병무청장은 사회복무요원 소집이 연기된 사람으로서 그 사유가 없어진 사람 등 대통령령으로 정하는 사람에 대\n하여는 제1항에도 불구하고 지방병무청장으로 하여금 따로 사회복무요원 소집을 하게 할 수 있다.<개정 2013. 6.\n4.>\n③ 제1항과 제2항에 따라 사회복무요원으로 소집된 사람에 대하여는 제55조에 따른 군사교육소집을 하며, 그 군사\n교육소집 기간은 복무기간에 산입한다.<개정 2013. 6. 4., 2016. 5. 29.>\n④ 지방병무청장은 복무기관의 장이 사회복무요원의 임무 부여에 활용할 수 있도록 사회복무요원이 다음 각 호의\n범죄로 인하여 형의 선고를 받은 경우 관련된 정보를 그 사회복무요원이 복무할 복무기관의 장에게 제공할 수 있으\n며, 정보 제공 절차 및 제공되는 정보의 범위는 병무청장이 정한다.<신설 2020. 12. 22., 2024. 1. 9.>\n1. 「아동ㆍ청소년의 성보호에 관한 법률」 제2조제2호, 제3호 및 제3호의2에 따른 범죄\n2. 「개인정보 보호법」 제70조부터 제73조까지에 따른 범죄\n3. 「특정강력범죄의 처벌에 관한 특례법」 제2조의 특정강력범죄\n4. 「형법」 제257조, 제258조, 제258조의2, 제259조부터 제265조까지에 따른 상해와 폭행의 죄, 제283조부터 제\n286조까지에 따른 협박의 죄\n5. 「마약류 관리에 관한 법률」 제58조, 제58조의2, 제59조부터 제64조까지, 제65조의2에 따른 범죄\n[전문개정 2009. 6. 9.]\n[제목개정 2013. 6. 4., 2020. 12. 22.]']
생성된 답변(정답): 병무

In [ ]:
# 생성된 Testset을 Pandas DataFrame으로 변환
eval_df = testset.to_pandas()

# import os

# SAVE_DIR = r"C:\Users\Playdata\SKN\SKN31-3rd-2Team\eval_results"
# os.makedirs(SAVE_DIR, exist_ok=True)

# eval_df["source"] = "neo4j"  # 나중에 출처별로 점수 나눠보기 위함
# eval_df.to_pickle(os.path.join(SAVE_DIR, "eval_testset_neo4j.pkl"))
# # CSV는 reference_contexts가 list라서 저장 시 깨짐 -> pickle 사용

In [38]:
eval_df.shape

(11, 8)

In [39]:
eval_df

,user_input,reference_contexts,reference,persona_name,query_style,query_length,synthesizer_name,source
0,대통령령에 따라 사회복무요원 소집이 연기된 사람이 다시 소집될 수 있는 경우는 무엇...,[① 지방병무청장은 사회복무요원 소집 순서가 결정된 사람을 대상으로 복무기관을 정\...,병무청장은 사회복무요원 소집이 연기된 사람으로서 그 사유가 없어진 사람 등 대통령령...,Military Incident Reporting Officer,MISSPELLED,LONG,single_hop_specific_query_synthesizer,qdrant
1,지방병무청장의 역할은 무엇인가요?,[① 국방부장관은 법 제58조 및 제59조에 따라 의무ㆍ법무ㆍ군종ㆍ수\n의 분야의 ...,지방병무청장은 입영 계획서에 따라 현역입영 통지서를 입영일 7일 전까지 대상자에게 ...,Military Incident Reporting Officer,WEB_SEARCH_LIKE,SHORT,single_hop_specific_query_synthesizer,qdrant
2,"군사경찰부대 뭐하는데, 사고나면 군사경찰부대가 어떻게 해야돼요, 그리고 사고속보는 ...",[① 사고가 발생하였을 때의 보고의 유형 및 보고계통은 다음 각 호와 같다. 다만 ...,"사고가 발생하면 사고속보는 제262조에 따라 군사경찰계통으로 보고해야 하고, 사고 ...",Military Personnel Administrator,POOR_GRAMMAR,LONG,single_hop_specific_query_synthesizer,qdrant
3,국가법령정보센터에서 군 복무 중인 군인과 군무원이 반드시 지켜야 하는 맹세와 관련된...,"[군에 복무 중인 군인, 근무 중인 군무원은 국가에 대하여 다음 각 호와 같이 맹세...",국가법령정보센터에서는 군에 복무 중인 군인과 근무 중인 군무원이 국가에 대하여 맹세...,Military Incident Reporting Officer,WEB_SEARCH_LIKE,LONG,single_hop_specific_query_synthesizer,qdrant
4,불침번이 근무 중 이상이 있을 때 당직계통에 어떻게 보고해야 합니까?,"[① 불침번은 건물 내ㆍ외를 완벽히 경계하여야 하며, 근무 중 지정된 위치를 떠나거...","불침번은 근무 중 이상이 있을 경우 필요한 조치를 하고, 당직계통의 상관에게 보고하...",Military Personnel Administrator,PERFECT_GRAMMAR,SHORT,single_hop_specific_query_synthesizer,qdrant
5,국방부장관 입영계획서 언제 보내야되? 그리고 급여나 재해보상 관련해서 국방부장관이 ...,[<1-hop>\n\n① 국방부장관은 법 제58조 및 제59조에 따라 의무ㆍ법무ㆍ군...,국방부장관은 입영기본 계획서를 입영일 40일 전까지 병무청장 및 해당 군 참모총장에...,NaN,NaN,NaN,multi_hop_specific_query_synthesizer,qdrant
6,국방부장관이 의무ㆍ법무ㆍ군종ㆍ수의장교 및 기본병과장교의 병적 편입과 관련하여 입영 ...,[<1-hop>\n\n① 국방부장관은 법 제58조 및 제59조에 따라 의무ㆍ법무ㆍ군...,"국방부장관은 의무ㆍ법무ㆍ군종ㆍ수의장교 및 기본병과장교의 병적 편입과 관련하여, 입영...",NaN,NaN,NaN,multi_hop_specific_query_synthesizer,qdrant
7,"국방부 관련 규정에 따라 성희롱 또는 성폭력 사고가 발생했을 때, 보고의 유형과 보...",[<1-hop>\n\n① 사고가 발생하였을 때의 보고의 유형 및 보고계통은 다음 각...,성희롱 또는 성폭력 사고가 발생했을 때에는 일반 사고와 달리 각 군 본부 성고충예방...,NaN,NaN,NaN,multi_hop_specific_query_synthesizer,qdrant
8,"국방부와 각 군 본부에서 성희롱 또는 성폭력 사고가 발생했을 때, 사고 보고의 유형...",[<1-hop>\n\n① 사고가 발생하였을 때의 보고의 유형 및 보고계통은 다음 각...,성희롱 또는 성폭력 사고가 발생했을 때에는 일반 사고와 달리 각 군 본부 성고충예방...,NaN,NaN,NaN,multi_hop_specific_query_synthesizer,qdrant
9,법제처에서 정한 규정에 따라 군인이나 군무원이 포로가 되었을 때 지켜야 하는 맹세 ...,"[<1-hop>\n\n군에 복무 중인 군인, 근무 중인 군무원은 국가에 대하여 다음...",군에 복무 중인 군인이나 근무 중인 군무원은 법제처에서 정한 규정에 따라 국가와 민...,NaN,NaN,NaN,multi_hop_specific_query_synthesizer,qdrant


In [40]:
row_idx = 0
q = eval_df.loc[row_idx, 'user_input']
resp = chain.invoke(q)


Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "CALL db.index.vector.queryNodes('article_vector_index', $top_k, $query_embedding)\nYIELD node, score\nRETURN node.id AS id, node.name AS name, node.description AS description, score\nORDER BY score DESC"


In [41]:
# resp
resp['response']

'사회복무요원 소집이 연기된 사람은 **그 연기 사유가 없어진 경우** 다시 별도로 소집될 수 있습니다.  \n\n근거: **병역법 시행령 제52조 제3호**  \n- “사회복무요원 소집이 연기된 사람으로서 **그 사유가 없어진 사람**”'

In [42]:
# resp
resp['retrieved_context']

['법 제29조제2항에 따라 사회복무요원 소집 순서에 따르지 않고 따로 사회복무\n요원 소집을 할 수 있는 사람은 다음 각 호와 같다. <개정 2010. 7. 21., 2011. 7. 14., 2013. 12. 4., 2016. 6. 14., 2016.\n11. 29., 2017. 9. 22., 2020. 6. 30., 2021. 10. 14.>\n1. 병역판정검사를 받고 그 해 소집을 희망하는 사람\n2. 학군 군간부후보생 또는 의무ㆍ법무ㆍ군종ㆍ수의사관후보생으로서 해당 병적에서 제적되어 소집할 사람\n3. 사회복무요원 소집이 연기된 사람으로서 그 사유가 없어진 사람\n4. 국외에서 귀국한 사람으로서 소집할 사람\n5. 사회복무요원 소집 기피의 죄를 범한 사람으로서 형사처분이 종료된 사람 및 공소시효가 완성된 사람\n6. 예술ㆍ체육요원, 공중보건의사, 병역판정전담의사, 공익법무관, 공중방역수의사, 전문연구요원 또는 산업기능요\n원으로의 편입이 취소되어 소집할 사람\n7. 제137조제7항에 따라 보충역으로 편입된 사람\n8. 대체역법 제25조제2항에 따라 대체역의 편입이 취소되어 소집할 사람\n9. 그 밖에 병무청장이 필요하다고 인정하는 사람\n[전문개정 2009. 12. 7.]\n[제목개정 2013. 12. 4.]',
 '① 법 제65조제11항 본문에 따라 사회복무요원으\n로 계속 복무하는 것이 적합하지 아니한 사람은 다음 각 호와 같다. <개정 2017. 5. 29.>\n1. 같은 질병이나 심신장애로 6개월 이상 지속적인 치료에도 불구하고 치유되지 아니하여 정상적인 직무 수행이 곤\n란한 사람\n2. 「정신건강증진 및 정신질환자 복지서비스 지원에 관한 법률」 제3조제1호의 정신질환자로서 근무시간 중 다른\n사람의 생명 또는 신체에 중대한 위해(危害)를 입힐 우려가 있는 사람\n3. 지능 정도가 현저히 낮거나 발달장애 등으로 다른 사람의 도움 없이는 단순한 직무 수행도 곤란한 사람\n4. 마약ㆍ대마ㆍ향정신성의약품 또는 알코올에 중독되어 치료 등을 받았으나 치유

In [43]:
# 모든 testset의 데이터데 대한 llm 응답과 retriever의 검색 겷과를 추가
response_list = []
retrieved_context_list = []

for user_input in eval_df['user_input']:
    resp = chain.invoke(user_input)
    response_list.append(resp["response"])
    retrieved_context_list.append(resp["retrieved_context"])

Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "CALL db.index.vector.queryNodes('article_vector_index', $top_k, $query_embedding)\nYIELD node, score\nRETURN node.id AS id, node.name AS name, node.description AS description, score\nORDER BY score DESC"
Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement.

In [44]:
print(len(response_list), len(retrieved_context_list))

11 11


In [45]:
response_list

['사회복무요원 소집이 연기된 사람은 **그 연기 사유가 없어졌을 때** 다시 소집될 수 있습니다.  \n근거: 병역법 시행령 제52조 제3호.',
 '정보가 부족해 답을 할 수없습니다.',
 '주어진 Context 기준으로만 답변드리면 다음과 같습니다.\n\n### 1) 군사경찰부대는 뭐 하는 곳인가요?\nContext에는 **군사경찰부대의 일반적인 업무 전체**가 자세히 나오지는 않습니다. 다만 사고와 관련해서는 다음 역할이 확인됩니다.\n\n- **사고 발생 시 인지 즉시 신고를 받는 대상**\n- **사고속보를 군사경찰계통으로 보고하는 체계의 일부**\n- **국방부조사본부가 관할 변경되거나 수사 조정·통제를 할 때 수사인력 지원에 협조하는 대상**\n\n즉, Context상으로는 **사고 신고 접수, 사고속보 보고, 수사 지원과 관련된 역할**이 확인됩니다.\n\n### 2) 사고가 나면 군사경찰부대가 어떻게 해야 하나요?\nContext에 따르면:\n\n- **사고 발생 시 당사자 또는 당사자로부터 보고를 받은 사람은 관할 군사경찰부대에 인지 즉시 신고해야 합니다.**\n- **사고속보는 제262조에 따라 군사경찰계통으로 보고합니다.**\n- 또한 **중대 사건·사고의 관할이 국방부조사본부로 변경되는 경우**, 국방부조사본부장은 수사인력 지원을 요청할 수 있고 **각 군 군사경찰실(단)장은 특별한 사유가 없는 한 이를 지원해야 합니다.**\n\n즉, 사고가 나면 군사경찰부대는 **신고를 받고, 사고속보를 처리하며, 필요시 수사 지원에 협조**해야 합니다.\n\n### 3) “사고속보는 군사경찰계통으로 보고한다”는 뜻은 무엇인가요?\nContext에 따르면 **사고속보는 제262조에 따라 군사경찰계통으로 보고**한다고 되어 있습니다.\n\n이 뜻은:\n- 사고와 관련한 **긴급한 초기 정보**를\n- **군사경찰의 보고 라인(계통)**을 통해\n- 정해진 절차에 따라 전달한다는 의미입니다.\n\n즉, 일반 행정라인이 아니라 **군사경찰 조직을 통한 보

In [46]:
retrieved_context_list

[['법 제29조제2항에 따라 사회복무요원 소집 순서에 따르지 않고 따로 사회복무\n요원 소집을 할 수 있는 사람은 다음 각 호와 같다. <개정 2010. 7. 21., 2011. 7. 14., 2013. 12. 4., 2016. 6. 14., 2016.\n11. 29., 2017. 9. 22., 2020. 6. 30., 2021. 10. 14.>\n1. 병역판정검사를 받고 그 해 소집을 희망하는 사람\n2. 학군 군간부후보생 또는 의무ㆍ법무ㆍ군종ㆍ수의사관후보생으로서 해당 병적에서 제적되어 소집할 사람\n3. 사회복무요원 소집이 연기된 사람으로서 그 사유가 없어진 사람\n4. 국외에서 귀국한 사람으로서 소집할 사람\n5. 사회복무요원 소집 기피의 죄를 범한 사람으로서 형사처분이 종료된 사람 및 공소시효가 완성된 사람\n6. 예술ㆍ체육요원, 공중보건의사, 병역판정전담의사, 공익법무관, 공중방역수의사, 전문연구요원 또는 산업기능요\n원으로의 편입이 취소되어 소집할 사람\n7. 제137조제7항에 따라 보충역으로 편입된 사람\n8. 대체역법 제25조제2항에 따라 대체역의 편입이 취소되어 소집할 사람\n9. 그 밖에 병무청장이 필요하다고 인정하는 사람\n[전문개정 2009. 12. 7.]\n[제목개정 2013. 12. 4.]',
  '① 법 제65조제11항 본문에 따라 사회복무요원으\n로 계속 복무하는 것이 적합하지 아니한 사람은 다음 각 호와 같다. <개정 2017. 5. 29.>\n1. 같은 질병이나 심신장애로 6개월 이상 지속적인 치료에도 불구하고 치유되지 아니하여 정상적인 직무 수행이 곤\n란한 사람\n2. 「정신건강증진 및 정신질환자 복지서비스 지원에 관한 법률」 제3조제1호의 정신질환자로서 근무시간 중 다른\n사람의 생명 또는 신체에 중대한 위해(危害)를 입힐 우려가 있는 사람\n3. 지능 정도가 현저히 낮거나 발달장애 등으로 다른 사람의 도움 없이는 단순한 직무 수행도 곤란한 사람\n4. 마약ㆍ대마ㆍ향정신성의약품 또는 알코올에 중독되어 치료 등을 받았으나 

In [47]:
#
# eval_df에 컬럼으로 추가
#
eval_df['response'] = response_list
eval_df['retrieved_contexts'] = retrieved_context_list
eval_df

,user_input,reference_contexts,reference,persona_name,query_style,query_length,synthesizer_name,source,response,retrieved_contexts
0,대통령령에 따라 사회복무요원 소집이 연기된 사람이 다시 소집될 수 있는 경우는 무엇...,[① 지방병무청장은 사회복무요원 소집 순서가 결정된 사람을 대상으로 복무기관을 정\...,병무청장은 사회복무요원 소집이 연기된 사람으로서 그 사유가 없어진 사람 등 대통령령...,Military Incident Reporting Officer,MISSPELLED,LONG,single_hop_specific_query_synthesizer,qdrant,사회복무요원 소집이 연기된 사람은 **그 연기 사유가 없어졌을 때** 다시 소집될 ...,[법 제29조제2항에 따라 사회복무요원 소집 순서에 따르지 않고 따로 사회복무\n요...
1,지방병무청장의 역할은 무엇인가요?,[① 국방부장관은 법 제58조 및 제59조에 따라 의무ㆍ법무ㆍ군종ㆍ수\n의 분야의 ...,지방병무청장은 입영 계획서에 따라 현역입영 통지서를 입영일 7일 전까지 대상자에게 ...,Military Incident Reporting Officer,WEB_SEARCH_LIKE,SHORT,single_hop_specific_query_synthesizer,qdrant,정보가 부족해 답을 할 수없습니다.,"[지방병무청장은 병무지청장, 병역판정전담의사 또는 군의관이 법 또는 이 영에 따라 ..."
2,"군사경찰부대 뭐하는데, 사고나면 군사경찰부대가 어떻게 해야돼요, 그리고 사고속보는 ...",[① 사고가 발생하였을 때의 보고의 유형 및 보고계통은 다음 각 호와 같다. 다만 ...,"사고가 발생하면 사고속보는 제262조에 따라 군사경찰계통으로 보고해야 하고, 사고 ...",Military Personnel Administrator,POOR_GRAMMAR,LONG,single_hop_specific_query_synthesizer,qdrant,주어진 Context 기준으로만 답변드리면 다음과 같습니다.\n\n### 1) 군사...,[국방부 사고대책반의 임무는 다음 각 호와 같다.\n1. 사고발생 초기 상황유지 및...
3,국가법령정보센터에서 군 복무 중인 군인과 군무원이 반드시 지켜야 하는 맹세와 관련된...,"[군에 복무 중인 군인, 근무 중인 군무원은 국가에 대하여 다음 각 호와 같이 맹세...",국가법령정보센터에서는 군에 복무 중인 군인과 근무 중인 군무원이 국가에 대하여 맹세...,Military Incident Reporting Officer,WEB_SEARCH_LIKE,LONG,single_hop_specific_query_synthesizer,qdrant,정보가 부족해 답을 할 수없습니다.,"[군에 복무 중인 군인, 근무 중인 군무원은 국가에 대하여 다음 각 호와 같이 맹세..."
4,불침번이 근무 중 이상이 있을 때 당직계통에 어떻게 보고해야 합니까?,"[① 불침번은 건물 내ㆍ외를 완벽히 경계하여야 하며, 근무 중 지정된 위치를 떠나거...","불침번은 근무 중 이상이 있을 경우 필요한 조치를 하고, 당직계통의 상관에게 보고하...",Military Personnel Administrator,PERFECT_GRAMMAR,SHORT,single_hop_specific_query_synthesizer,qdrant,"불침번은 근무 중 이상이 있을 경우 필요한 조치를 하고, **당직계통의 상관에게 보...",[불침번은 통상 저녁점호 직후부터 다음날 기상시까지 생활관 또는 건물단위로\n지휘관...
5,국방부장관 입영계획서 언제 보내야되? 그리고 급여나 재해보상 관련해서 국방부장관이 ...,[<1-hop>\n\n① 국방부장관은 법 제58조 및 제59조에 따라 의무ㆍ법무ㆍ군...,국방부장관은 입영기본 계획서를 입영일 40일 전까지 병무청장 및 해당 군 참모총장에...,NaN,NaN,NaN,multi_hop_specific_query_synthesizer,qdrant,정보가 부족해 답을 할 수없습니다.,[① 국방부장관은 제8조제1항에 따른 급여의 결정 및 지급에 관한 권한을\n대통령령...
6,국방부장관이 의무ㆍ법무ㆍ군종ㆍ수의장교 및 기본병과장교의 병적 편입과 관련하여 입영 ...,[<1-hop>\n\n① 국방부장관은 법 제58조 및 제59조에 따라 의무ㆍ법무ㆍ군...,"국방부장관은 의무ㆍ법무ㆍ군종ㆍ수의장교 및 기본병과장교의 병적 편입과 관련하여, 입영...",NaN,NaN,NaN,multi_hop_specific_query_synthesizer,qdrant,국방부장관의 주요 역할은 다음과 같습니다.\n\n1. **입영기본 계획서 송부**\...,[① 국방부장관은 법 제58조 및 제59조에 따라 의무ㆍ법무ㆍ군종ㆍ수\n의 분야의 ...
7,"국방부 관련 규정에 따라 성희롱 또는 성폭력 사고가 발생했을 때, 보고의 유형과 보...",[<1-hop>\n\n① 사고가 발생하였을 때의 보고의 유형 및 보고계통은 다음 각...,성희롱 또는 성폭력 사고가 발생했을 때에는 일반 사고와 달리 각 군 본부 성고충예방...,NaN,NaN,NaN,multi_hop_specific_query_synthesizer,qdrant,"부대관리훈령 제260조에 따르면, **성희롱ㆍ성폭력 사고**는 원칙적으로 **각 군...",[① 모든 군인 및 군무원은 성희롱ㆍ성폭력 관련 사실을 직접 목격하거나 타인에게 듣...
8,"국방부와 각 군 본부에서 성희롱 또는 성폭력 사고가 발생했을 때, 사고 보고의 유형...",[<1-hop>\n\n① 사고가 발생하였을 때의 보고의 유형 및 보고계통은 다음 각...,성희롱 또는 성폭력 사고가 발생했을 때에는 일반 사고와 달리 각 군 본부 성고충예방...,NaN,NaN,NaN,multi_hop_specific_query_synthesizer,qdrant,"부대관리훈령 제260조에 따르면, **성희롱ㆍ성폭력은 각 군 본부 성고충예방대응센터...",[① 모든 군인 및 군무원은 성희롱ㆍ성폭력 관련 사실을 직접 목격하거나 타인에게 듣...
9,법제처에서 정한 규정에 따라 군인이나 군무원이 포로가 되었을 때 지켜야 하는 맹세 ...,"[<1-hop>\n\n군에 복무 중인 군인, 근무 중인 군무원은 국가에 대하여 다음...",군에 복무 중인 군인이나 근무 중인 군무원은 법제처에서 정한 규정에 따라 국가와 민...,NaN,NaN,NaN,multi_hop_specific_query_synthesizer,qdrant,정보가 부족해 답을 할 수없습니다.,"[① 국가기관, 지방자치단체의 장 또는 고용주는 소속 공무원이나 임직원이 징집ㆍ소집..."


In [48]:
# eval_df를 ragas의 평가데이터셋 타입으로 변환
from ragas import EvaluationDataset
eval_dataset = EvaluationDataset.from_pandas(
    eval_df[["user_input", "retrieved_contexts", "response", "reference"]]
)
eval_dataset

EvaluationDataset(features=['user_input', 'retrieved_contexts', 'response', 'reference'], len=11)

In [49]:
#
# 평가
#
from ragas.metrics import (
    LLMContextRecall,
    LLMContextPrecisionWithReference,
    Faithfulness,
    AnswerRelevancy,
)
from ragas import evaluate

C:\Users\Playdata\AppData\Local\Temp\ipykernel_30664\3378832119.py:4: DeprecationWarning: Importing LLMContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextRecall
  from ragas.metrics import (
C:\Users\Playdata\AppData\Local\Temp\ipykernel_30664\3378832119.py:4: DeprecationWarning: Importing LLMContextPrecisionWithReference from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextPrecisionWithReference
  from ragas.metrics import (
C:\Users\Playdata\AppData\Local\Temp\ipykernel_30664\3378832119.py:4: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import (
C:\User

In [50]:
# 평가할 때 사용할 LLM embedding 모델
eval_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1"))
eval_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-large"))

metrics = [
    LLMContextRecall(llm=eval_llm),
    LLMContextPrecisionWithReference(llm=eval_llm),
    Faithfulness(llm=eval_llm),
    AnswerRelevancy(llm=eval_llm, embeddings=eval_embeddings)

]
eval_result = evaluate(dataset=eval_dataset, metrics=metrics)

C:\Users\Playdata\AppData\Local\Temp\ipykernel_30664\384477035.py:2: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  eval_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1"))
C:\Users\Playdata\AppData\Local\Temp\ipykernel_30664\384477035.py:3: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  eval_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-large"))
Evaluating:  14%|█▎        | 6/44 [00:05<00:24,  1.54it/s]LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.

In [51]:
eval_result

{'context_recall': 0.6640, 'llm_context_precision_with_reference': 0.5667, 'faithfulness': 0.2143, 'answer_relevancy': 0.2296}

In [52]:
result_df = eval_result.to_pandas()
result_df

,user_input,retrieved_contexts,response,reference,context_recall,llm_context_precision_with_reference,faithfulness,answer_relevancy
0,대통령령에 따라 사회복무요원 소집이 연기된 사람이 다시 소집될 수 있는 경우는 무엇...,[법 제29조제2항에 따라 사회복무요원 소집 순서에 따르지 않고 따로 사회복무\n요...,사회복무요원 소집이 연기된 사람은 **그 연기 사유가 없어졌을 때** 다시 소집될 ...,병무청장은 사회복무요원 소집이 연기된 사람으로서 그 사유가 없어진 사람 등 대통령령...,1.000000,1.000000,0.5,0.539641
1,지방병무청장의 역할은 무엇인가요?,"[지방병무청장은 병무지청장, 병역판정전담의사 또는 군의관이 법 또는 이 영에 따라 ...",정보가 부족해 답을 할 수없습니다.,지방병무청장은 입영 계획서에 따라 현역입영 통지서를 입영일 7일 전까지 대상자에게 ...,0.000000,0.000000,0.0,0.000000
2,"군사경찰부대 뭐하는데, 사고나면 군사경찰부대가 어떻게 해야돼요, 그리고 사고속보는 ...",[국방부 사고대책반의 임무는 다음 각 호와 같다.\n1. 사고발생 초기 상황유지 및...,주어진 Context 기준으로만 답변드리면 다음과 같습니다.\n\n### 1) 군사...,"사고가 발생하면 사고속보는 제262조에 따라 군사경찰계통으로 보고해야 하고, 사고 ...",1.000000,0.416667,NaN,0.843799
3,국가법령정보센터에서 군 복무 중인 군인과 군무원이 반드시 지켜야 하는 맹세와 관련된...,"[군에 복무 중인 군인, 근무 중인 군무원은 국가에 대하여 다음 각 호와 같이 맹세...",정보가 부족해 답을 할 수없습니다.,국가법령정보센터에서는 군에 복무 중인 군인과 근무 중인 군무원이 국가에 대하여 맹세...,1.000000,1.000000,0.0,0.000000
4,불침번이 근무 중 이상이 있을 때 당직계통에 어떻게 보고해야 합니까?,[불침번은 통상 저녁점호 직후부터 다음날 기상시까지 생활관 또는 건물단위로\n지휘관...,"불침번은 근무 중 이상이 있을 경우 필요한 조치를 하고, **당직계통의 상관에게 보...","불침번은 근무 중 이상이 있을 경우 필요한 조치를 하고, 당직계통의 상관에게 보고하...",1.000000,0.450000,1.0,0.810833
5,국방부장관 입영계획서 언제 보내야되? 그리고 급여나 재해보상 관련해서 국방부장관이 ...,[① 국방부장관은 제8조제1항에 따른 급여의 결정 및 지급에 관한 권한을\n대통령령...,정보가 부족해 답을 할 수없습니다.,국방부장관은 입영기본 계획서를 입영일 40일 전까지 병무청장 및 해당 군 참모총장에...,0.500000,0.250000,0.0,0.000000
6,국방부장관이 의무ㆍ법무ㆍ군종ㆍ수의장교 및 기본병과장교의 병적 편입과 관련하여 입영 ...,[① 국방부장관은 법 제58조 및 제59조에 따라 의무ㆍ법무ㆍ군종ㆍ수\n의 분야의 ...,국방부장관의 주요 역할은 다음과 같습니다.\n\n1. **입영기본 계획서 송부**\...,"국방부장관은 의무ㆍ법무ㆍ군종ㆍ수의장교 및 기본병과장교의 병적 편입과 관련하여, 입영...",1.000000,0.833333,NaN,0.330904
7,"국방부 관련 규정에 따라 성희롱 또는 성폭력 사고가 발생했을 때, 보고의 유형과 보...",[① 모든 군인 및 군무원은 성희롱ㆍ성폭력 관련 사실을 직접 목격하거나 타인에게 듣...,"부대관리훈령 제260조에 따르면, **성희롱ㆍ성폭력 사고**는 원칙적으로 **각 군...",성희롱 또는 성폭력 사고가 발생했을 때에는 일반 사고와 달리 각 군 본부 성고충예방...,NaN,1.000000,NaN,0.000000
8,"국방부와 각 군 본부에서 성희롱 또는 성폭력 사고가 발생했을 때, 사고 보고의 유형...",[① 모든 군인 및 군무원은 성희롱ㆍ성폭력 관련 사실을 직접 목격하거나 타인에게 듣...,"부대관리훈령 제260조에 따르면, **성희롱ㆍ성폭력은 각 군 본부 성고충예방대응센터...",성희롱 또는 성폭력 사고가 발생했을 때에는 일반 사고와 달리 각 군 본부 성고충예방...,NaN,0.950000,NaN,0.000000
9,법제처에서 정한 규정에 따라 군인이나 군무원이 포로가 되었을 때 지켜야 하는 맹세 ...,"[① 국가기관, 지방자치단체의 장 또는 고용주는 소속 공무원이나 임직원이 징집ㆍ소집...",정보가 부족해 답을 할 수없습니다.,군에 복무 중인 군인이나 근무 중인 군무원은 법제처에서 정한 규정에 따라 국가와 민...,0.142857,0.333333,0.0,0.000000


In [53]:
driver.close()
